# Simuladores para Seleccion Natural como Algoritmo de Busqueda
**Lab25 - Arturo Bouzas** — *Aprendizaje y Comportamiento Adaptable I*

---
**Instrucciones:** Ejecuta **Runtime > Run all** para cargar todos los simuladores.  
Luego ejecuta la celda del simulador que quieras usar.

In [ ]:
# ============================================================
# IMPORTACIONES
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML
from scipy.stats import linregress
from matplotlib.patches import Rectangle
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# ============================================================
# SIMULADOR 1: ALGORITMO EVOLUTIVO MINIMO
# ============================================================

class EvolutionSimulator:
    def __init__(self, pop_size=50, mutation_rate=0.1, selection_strength=1.0):
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.selection_strength = selection_strength
        self.reset()

    def fitness_landscape(self, x, y, landscape_type='single_peak'):
        if landscape_type == 'single_peak':
            return np.exp(-((x-0.5)**2 + (y-0.5)**2) / 0.1)
        elif landscape_type == 'two_peaks':
            peak1 = np.exp(-((x-0.3)**2 + (y-0.3)**2) / 0.05)
            peak2 = np.exp(-((x-0.7)**2 + (y-0.7)**2) / 0.05)
            return np.maximum(peak1, peak2)
        else:
            return np.ones_like(x)

    def reset(self):
        self.population = np.random.rand(self.pop_size, 2)
        self.generation = 0
        self.history = {'mean_fitness': [], 'generation': []}

    def evaluate_fitness(self, landscape_type='single_peak'):
        x, y = self.population[:, 0], self.population[:, 1]
        return self.fitness_landscape(x, y, landscape_type)

    def selection(self, fitness):
        probs = fitness ** self.selection_strength
        probs /= probs.sum()
        idx = np.random.choice(self.pop_size, size=self.pop_size, p=probs, replace=True)
        return self.population[idx]

    def mutation(self, offspring):
        noise = np.random.normal(0, self.mutation_rate, offspring.shape)
        return np.clip(offspring + noise, 0, 1)

    def step(self, landscape_type='single_peak'):
        fitness = self.evaluate_fitness(landscape_type)
        self.history['mean_fitness'].append(fitness.mean())
        self.history['generation'].append(self.generation)
        self.population = self.mutation(self.selection(fitness))
        self.generation += 1
        return fitness

    def plot_state(self, landscape_type='single_peak', ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(8, 6))
        xg = np.linspace(0, 1, 100)
        X, Y = np.meshgrid(xg, xg)
        Z = self.fitness_landscape(X, Y, landscape_type)
        ax.contourf(X, Y, Z, levels=20, cmap='RdYlGn', alpha=0.6)
        ax.scatter(self.population[:, 0], self.population[:, 1],
                   c='blue', s=50, alpha=0.7, edgecolors='black')
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel('Rasgo X'); ax.set_ylabel('Rasgo Y')
        ax.set_title('Generacion ' + str(self.generation))
        return ax


def create_evolutionary_algorithm_interactive():
    sim = EvolutionSimulator()
    mut_sl = widgets.FloatSlider(value=0.05, min=0, max=0.3, step=0.01,
        description='Mutacion:', style={'description_width': '120px'})
    sel_sl = widgets.FloatSlider(value=2.0, min=0, max=5.0, step=0.1,
        description='Seleccion:', style={'description_width': '120px'})
    land_dd = widgets.Dropdown(options=['single_peak', 'two_peaks', 'flat'],
        value='single_peak', description='Paisaje:')
    step_btn  = widgets.Button(description="Siguiente Generacion")
    run10_btn = widgets.Button(description="Correr 10 Gen")
    reset_btn = widgets.Button(description="Reset")
    output = widgets.Output()

    info = widgets.HTML(
        "<div style='background-color:#e3f2fd;padding:10px;border-radius:5px;"
        "color:#1a1a1a;'>"
        "<b>Algoritmo Evolutivo Minimo</b><br><br>"
        "<b>Mutacion:</b> Cuanto varian los rasgos (exploracion)<br>"
        "<b>Seleccion:</b> Intensidad de la seleccion (explotacion)<br>"
        "<b>Paisaje:</b> Funcion de fitness del ambiente<br><br>"
        "Con sigma=0: no hay cambio | Con s=0: deriva aleatoria | "
        "Dos picos: riesgo de optimo local"
        "</div>"
    )

    def update_plot():
        with output:
            output.clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
            sim.plot_state(land_dd.value, ax=ax1)
            ax2.plot(sim.history['generation'], sim.history['mean_fitness'], 'b-')
            ax2.set_xlabel('Generacion'); ax2.set_ylabel('Fitness Promedio')
            ax2.set_title('Evolucion del Fitness'); ax2.grid(True)
            plt.tight_layout(); plt.show()

    def on_step(b):
        sim.mutation_rate = mut_sl.value
        sim.selection_strength = sel_sl.value
        sim.step(land_dd.value)
        update_plot()

    def on_run10(b):
        sim.mutation_rate = mut_sl.value
        sim.selection_strength = sel_sl.value
        for _ in range(10):
            sim.step(land_dd.value)
        update_plot()

    step_btn.on_click(on_step)
    run10_btn.on_click(on_run10)
    reset_btn.on_click(lambda b: (sim.reset(), update_plot()))

    controls = widgets.VBox([info, mut_sl, sel_sl, land_dd,
                              widgets.HBox([step_btn, run10_btn, reset_btn])])
    update_plot()
    display(widgets.HBox([controls, output]))


# ============================================================
# SIMULADOR 2: CRECIMIENTO DE CONTAGIO
# ============================================================

class ContagionSimulator:
    def __init__(self):
        self.reset()

    def reset(self):
        self.time = []
        self.I_history = []

    def simulate_contagion(self, I0, R, days):
        self.reset()
        I = float(I0)
        for day in range(days):
            self.time.append(day)
            self.I_history.append(I)
            I = I + R * I

    def plot_contagion(self, axes, R):
        ax1, ax2 = axes

        ax1.clear()
        ax1.plot(self.time, self.I_history, 'r-', linewidth=2.5, label='I(t)')
        ax1.set_xlabel('Dias', fontsize=12)
        ax1.set_ylabel('Infectados', fontsize=12)
        ax1.set_title('Escala lineal  (R = ' + str(round(R, 2)) + ')',
                      fontsize=13, fontweight='bold')
        ax1.legend(); ax1.grid(True, alpha=0.3)
        eq_text = 'I(t+1) = I(t) x (1 + ' + str(round(R, 2)) + ')'
        ax1.text(0.98, 0.05, eq_text, transform=ax1.transAxes,
                 ha='right', va='bottom', fontsize=11,
                 bbox=dict(boxstyle='round', facecolor='#fff9c4',
                           alpha=0.8, edgecolor='#aaa'))

        ax2.clear()
        I_arr = np.array(self.I_history)
        I_pos = np.where(I_arr > 0, I_arr, np.nan)
        ax2.plot(self.time, I_pos, 'b-', linewidth=2.5, label='I(t)')
        ax2.set_yscale('log')
        ax2.set_xlabel('Dias', fontsize=12)
        ax2.set_ylabel('Infectados (log)', fontsize=12)
        ax2.set_title('Escala logaritmica  (linea recta = crecimiento exponencial)',
                      fontsize=11, fontweight='bold')
        ax2.legend(); ax2.grid(True, alpha=0.3, which='both')
        if R > 0:
            td = np.log(2) / np.log(1 + R)
            ax2.text(0.98, 0.05, 'T2 = ' + str(round(td, 1)) + ' dias',
                     transform=ax2.transAxes, ha='right', va='bottom', fontsize=11,
                     bbox=dict(boxstyle='round', facecolor='#e3f2fd',
                               alpha=0.8, edgecolor='#aaa'))


def create_contagion_interactive():
    sim = ContagionSimulator()

    I0_sl = widgets.IntSlider(value=10, min=1, max=100, step=1,
        description='I0 (iniciales):', style={'description_width': '150px'})
    R_sl = widgets.FloatSlider(value=0.3, min=0.0, max=1.0, step=0.05,
        description='R (tasa contagio):', style={'description_width': '150px'},
        readout_format='.2f')
    days_sl = widgets.IntSlider(value=30, min=5, max=60, step=5,
        description='Dias a simular:', style={'description_width': '150px'})
    reset_btn = widgets.Button(description="Reset", button_style='danger')
    output = widgets.Output()

    info = widgets.HTML(
        "<div style='background-color:#e3f2fd;padding:10px;border-radius:5px;"
        "margin:10px 0;color:#1a1a1a;'>"
        "<b>Modelo Simple de Contagio</b><br><br>"
        "<b>Ecuacion en diferencia:</b><br>"
        "&nbsp;I(t+1) = I(t) + R*I(t) = I(t)*(1+R)<br><br>"
        "<b>R</b>: nuevos infectados por infectado por dia<br>"
        "COVID-19: R aprox 0.2-0.3 | Sarampion: R aprox 0.5-0.8<br><br>"
        "Panel derecho: escala log muestra el crecimiento exponencial como linea recta."
        "</div>"
    )

    def update_plot(*args):
        with output:
            output.clear_output(wait=True)
            sim.simulate_contagion(I0=I0_sl.value, R=R_sl.value, days=days_sl.value)
            fig, axes = plt.subplots(1, 2, figsize=(14, 5))
            sim.plot_contagion(axes, R=R_sl.value)
            plt.tight_layout(); plt.show()
            final = sim.I_history[-1]
            print("Infectados iniciales:", I0_sl.value)
            print("Infectados al dia " + str(days_sl.value) + ": " + str(round(final)))
            print("Factor de multiplicacion: " + str(round(final / I0_sl.value, 1)) + "x")
            if R_sl.value > 0:
                td = np.log(2) / np.log(1 + R_sl.value)
                print("Tiempo de duplicacion: " + str(round(td, 2)) + " dias")
            else:
                print("Con R=0 no hay duplicacion.")

    def on_reset(b):
        I0_sl.value = 10; R_sl.value = 0.3; days_sl.value = 30
        update_plot()

    for sl in [I0_sl, R_sl, days_sl]:
        sl.observe(update_plot, names='value')
    reset_btn.on_click(on_reset)

    controls = widgets.VBox([info, I0_sl, R_sl, days_sl, reset_btn],
        layout=widgets.Layout(min_width='380px', max_width='520px'))
    update_plot()
    display(widgets.HBox([controls, output]))


# ============================================================
# SIMULADOR 3: VISUALIZANDO COVARIANZA
# ============================================================

class CovarianceVisualizer:
    def __init__(self, n_points=50):
        self.n_points = n_points
        self.generate_data(correlation=0.7)

    def generate_data(self, correlation=0.7):
        mean = [0.5, 0.5]
        cov_matrix = [[0.1, correlation * 0.1], [correlation * 0.1, 0.1]]
        data = np.random.multivariate_normal(mean, cov_matrix, self.n_points)
        self.z = np.clip(data[:, 0], 0, 1)
        self.w = np.clip(data[:, 1], 0, 1)

    def compute_covariance(self):
        z_mean = np.mean(self.z); w_mean = np.mean(self.w)
        z_dev = self.z - z_mean; w_dev = self.w - w_mean
        products = z_dev * w_dev
        return np.mean(products), z_mean, w_mean, z_dev, w_dev, products

    def plot_covariance(self, ax, show_q=True, show_m=True, show_c=False):
        cov, z_mean, w_mean, z_dev, w_dev, products = self.compute_covariance()
        ax.clear()
        if show_q:
            ax.add_patch(Rectangle((z_mean, w_mean), 1-z_mean, 1-w_mean,
                facecolor='green', alpha=0.1, label='++ (Cov>0)'))
            ax.add_patch(Rectangle((0, 0), z_mean, w_mean,
                facecolor='green', alpha=0.1))
            ax.add_patch(Rectangle((0, w_mean), z_mean, 1-w_mean,
                facecolor='red', alpha=0.1, label='-+ (Cov<0)'))
            ax.add_patch(Rectangle((z_mean, 0), 1-z_mean, w_mean,
                facecolor='red', alpha=0.1))
        if show_m:
            ax.axvline(z_mean, color='blue', linestyle='--', linewidth=2,
                alpha=0.7, label='z_mean=' + str(round(z_mean, 2)))
            ax.axhline(w_mean, color='red', linestyle='--', linewidth=2,
                alpha=0.7, label='w_mean=' + str(round(w_mean, 2)))
        if show_c:
            sc = ax.scatter(self.z, self.w, c=products, cmap='RdYlGn',
                s=100, alpha=0.7, edgecolors='black', linewidths=1)
            plt.colorbar(sc, ax=ax, label='Contribucion a Cov')
        else:
            ax.scatter(self.z, self.w, s=100, alpha=0.6,
                edgecolors='black', linewidths=1, color='steelblue')
        sl, ic, rv, *_ = linregress(self.z, self.w)
        lx = np.array([0, 1])
        ax.plot(lx, sl * lx + ic, 'k-', linewidth=2, alpha=0.5,
                label='r=' + str(round(rv, 2)))
        ax.set_xlabel('Rasgo (z)', fontsize=12)
        ax.set_ylabel('Fitness (w)', fontsize=12)
        ax.set_title('Cov(w,z) = ' + str(round(cov, 4)), fontsize=14, fontweight='bold')
        ax.set_xlim(-0.05, 1.05); ax.set_ylim(-0.05, 1.05)
        ax.legend(loc='best', fontsize=9); ax.grid(True, alpha=0.3)
        ax.set_aspect('equal')


def create_covariance_interactive():
    viz = CovarianceVisualizer(n_points=50)

    corr_sl = widgets.FloatSlider(value=0.7, min=-1.0, max=1.0, step=0.1,
        description='Correlacion:', style={'description_width': '120px'},
        readout_format='.1f')
    n_sl = widgets.IntSlider(value=50, min=20, max=200, step=10,
        description='N organismos:', style={'description_width': '120px'})
    sq_cb = widgets.Checkbox(value=True,  description='Mostrar cuadrantes')
    sm_cb = widgets.Checkbox(value=True,  description='Mostrar medias')
    sc_cb = widgets.Checkbox(value=False, description='Colorear por contribucion')
    regen_btn = widgets.Button(description="Nueva Poblacion", button_style='info')
    output = widgets.Output()

    info = widgets.HTML(
        "<div style='background-color:#fff3cd;padding:10px;border-radius:5px;"
        "margin:10px 0;color:#1a1a1a;'>"
        "<b>Que es la Covarianza?</b><br><br>"
        "Cov(w,z) = Promedio de [(w - w_mean) x (z - z_mean)]<br><br>"
        "<b>Cuadrantes VERDES:</b> ambos arriba o ambos abajo de la media (contribucion POSITIVA)<br>"
        "<b>Cuadrantes ROJOS:</b> uno arriba, otro abajo (contribucion NEGATIVA)<br><br>"
        "Si Cov &gt; 0: rasgo z AUMENTARA &nbsp;|&nbsp; "
        "Si Cov &lt; 0: rasgo z DISMINUIRA &nbsp;|&nbsp; "
        "Si Cov = 0: sin cambio"
        "</div>"
    )

    def update_plot():
        with output:
            output.clear_output(wait=True)
            fig, ax = plt.subplots(figsize=(8, 8))
            viz.plot_covariance(ax, sq_cb.value, sm_cb.value, sc_cb.value)
            plt.tight_layout(); plt.show()
            cov, z_mean, w_mean, *_ = viz.compute_covariance()
            q1 = int(np.sum((viz.z > z_mean) & (viz.w > w_mean)))
            q2 = int(np.sum((viz.z < z_mean) & (viz.w > w_mean)))
            q3 = int(np.sum((viz.z < z_mean) & (viz.w < w_mean)))
            q4 = int(np.sum((viz.z > z_mean) & (viz.w < w_mean)))
            print("Cov(w,z) =", round(cov, 4),
                  " | z_mean =", round(z_mean, 3),
                  " | w_mean =", round(w_mean, 3))
            print("++ (verde sup-der):", q1,
                  " -- (verde inf-izq):", q3,
                  " -+ (rojo sup-izq):", q2,
                  " +- (rojo inf-der):", q4)
            if cov > 0.01:
                print("=> Dz > 0  (rasgo AUMENTA)")
            elif cov < -0.01:
                print("=> Dz < 0  (rasgo DISMINUYE)")
            else:
                print("=> Dz = 0  (sin cambio direccional)")

    corr_sl.observe(lambda c: (viz.generate_data(corr_sl.value), update_plot()), names='value')
    n_sl.observe(lambda c: (setattr(viz, 'n_points', n_sl.value),
                             viz.generate_data(corr_sl.value), update_plot()), names='value')
    regen_btn.on_click(lambda b: (viz.generate_data(corr_sl.value), update_plot()))
    for cb in [sq_cb, sm_cb, sc_cb]:
        cb.observe(lambda c: update_plot(), names='value')

    controls = widgets.VBox([info, corr_sl, n_sl, sq_cb, sm_cb, sc_cb, regen_btn])
    update_plot()
    display(widgets.HBox([controls, output]))


# ============================================================
# SIMULADOR 4: ECUACION DE CAMBIO EVOLUTIVO
# ============================================================

class EvolutionaryChangeSimulator:
    def __init__(self, pop_size=100):
        self.pop_size = pop_size
        self.reset_population()

    def reset_population(self, z_mean=0.5, z_std=0.15):
        self.generation = 0
        self.z = np.clip(np.random.normal(z_mean, z_std, self.pop_size), 0, 1)
        self.w = None
        self.history = {
            'generation': [0],
            'z_mean': [np.mean(self.z)],
            'z_std': [np.std(self.z)],
            'delta_z': [0],
            'cov_wz': [0],
            'w_mean': [0]
        }

    def fitness_function(self, z, fitness_type='linear_positive', params=None):
        if params is None:
            params = {}
        if fitness_type == 'linear_positive':
            w = params.get('a', 0.5) + params.get('b', 1.0) * z
        elif fitness_type == 'linear_negative':
            w = params.get('a', 1.5) + params.get('b', -1.0) * z
        elif fitness_type == 'quadratic':
            z_opt = params.get('z_opt', 0.6)
            width = params.get('width', 0.3)
            w = params.get('a', 2.0) - ((z - z_opt) / width) ** 2
        else:
            w = np.ones_like(z)
        return np.maximum(w, 0.01)

    def compute_evolution_equation(self):
        z_mean = np.mean(self.z)
        w_mean = np.mean(self.w)
        cov_wz = np.mean((self.z - z_mean) * (self.w - w_mean))
        delta_z = cov_wz / w_mean if w_mean > 0 else 0
        return {'z_mean': z_mean, 'w_mean': w_mean,
                'cov_wz': cov_wz, 'delta_z_predicted': delta_z}

    def selection_step(self):
        stats = self.compute_evolution_equation()
        probs = self.w / np.sum(self.w)
        idx = np.random.choice(self.pop_size, size=self.pop_size, p=probs, replace=True)
        z_orig_mean = stats['z_mean']
        self.z = np.clip(self.z[idx] + np.random.normal(0, 0.02, self.pop_size), 0, 1)
        self.generation += 1
        actual_delta = np.mean(self.z) - z_orig_mean
        self.history['generation'].append(self.generation)
        self.history['z_mean'].append(np.mean(self.z))
        self.history['z_std'].append(np.std(self.z))
        self.history['delta_z'].append(actual_delta)
        self.history['cov_wz'].append(stats['cov_wz'])
        self.history['w_mean'].append(stats['w_mean'])
        return stats['delta_z_predicted'], actual_delta

    def plot_current_state(self, fitness_type, params, axes):
        self.w = self.fitness_function(self.z, fitness_type, params)
        stats = self.compute_evolution_equation()

        ax1 = axes[0]; ax1.clear()
        ax1.scatter(self.z, self.w, s=50, alpha=0.6,
                    edgecolors='black', linewidths=0.5)
        ax1.axvline(stats['z_mean'], color='blue', linestyle='--', linewidth=2,
                    label='z_mean=' + str(round(stats['z_mean'], 3)))
        ax1.axhline(stats['w_mean'], color='red', linestyle='--', linewidth=2,
                    label='w_mean=' + str(round(stats['w_mean'], 3)))
        if len(self.z) > 1:
            sl, ic, rv, *_ = linregress(self.z, self.w)
            lz = np.linspace(0, 1, 100)
            ax1.plot(lz, sl * lz + ic, 'g-', linewidth=2, alpha=0.7,
                     label='r=' + str(round(rv, 2)))
        ax1.set_xlabel('Rasgo (z)'); ax1.set_ylabel('Fitness (w)')
        ax1.set_title('Gen ' + str(self.generation) +
                      ': Cov(w,z)=' + str(round(stats['cov_wz'], 4)),
                      fontweight='bold')
        ax1.legend(); ax1.grid(True, alpha=0.3)

        ax2 = axes[1]; ax2.clear()
        ax2.hist(self.z, bins=20, alpha=0.6, color='steelblue',
                 edgecolor='black', density=True,
                 label='Gen ' + str(self.generation))
        ax2.axvline(stats['z_mean'], color='blue', linewidth=2,
                    label='z_mean=' + str(round(stats['z_mean'], 3)))
        if stats['cov_wz'] != 0:
            pm = stats['z_mean'] + stats['delta_z_predicted']
            ax2.axvline(pm, color='red', linestyle='--', linewidth=2,
                        label='Predicho=' + str(round(pm, 3)))
        ax2.set_xlabel('Rasgo (z)'); ax2.set_ylabel('Densidad')
        ax2.set_title('Distribucion del Rasgo', fontweight='bold')
        ax2.legend(); ax2.grid(True, alpha=0.3, axis='y')

        ax3 = axes[2]; ax3.clear()
        if len(self.history['generation']) > 1:
            ga = np.array(self.history['generation'])
            zm = np.array(self.history['z_mean'])
            zs = np.array(self.history['z_std'])
            ax3.plot(ga, zm, 'bo-', linewidth=2, markersize=6, label='z_mean')
            ax3.fill_between(ga, zm - zs, zm + zs, alpha=0.2, color='blue')
            ax3.set_xlabel('Generacion'); ax3.set_ylabel('z_mean')
            ax3.set_title('Evolucion Temporal', fontweight='bold')
            ax3.legend(); ax3.grid(True, alpha=0.3)
        else:
            ax3.text(0.5, 0.5, 'Ejecuta generaciones\npara ver evolucion',
                     ha='center', va='center', fontsize=12,
                     transform=ax3.transAxes)


def create_evolutionary_change_interactive():
    sim = EvolutionaryChangeSimulator(pop_size=100)

    fit_dd = widgets.Dropdown(
        options=[
            ('Seleccion Positiva (z alto da mas descendientes)', 'linear_positive'),
            ('Seleccion Negativa (z bajo da mas descendientes)', 'linear_negative'),
            ('Optimo Intermedio (seleccion estabilizadora)',     'quadratic'),
            ('Sin Seleccion (deriva)',                           'flat')
        ],
        value='linear_positive',
        description='Tipo de seleccion:',
        style={'description_width': '150px'}
    )

    la_sl = widgets.FloatSlider(value=0.5, min=0.0, max=2.0, step=0.1,
        description='Intercepto (a):', style={'description_width': '150px'})
    lb_sl = widgets.FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1,
        description='Pendiente (b):', style={'description_width': '150px'})
    qo_sl = widgets.FloatSlider(value=0.6, min=0.0, max=1.0, step=0.05,
        description='z optimo:', style={'description_width': '150px'})
    qw_sl = widgets.FloatSlider(value=0.3, min=0.1, max=0.5, step=0.05,
        description='Ancho del pico:', style={'description_width': '150px'})

    step_btn  = widgets.Button(description="Siguiente Generacion", button_style='primary')
    run10_btn = widgets.Button(description="Correr 10 Gen",        button_style='success')
    run50_btn = widgets.Button(description="Correr 50 Gen",        button_style='warning')
    reset_btn = widgets.Button(description="Reset",                button_style='danger')
    output = widgets.Output()

    info = widgets.HTML(
        "<div style='background-color:#e8f5e9;padding:12px;border-radius:5px;"
        "margin:10px 0;color:#1a1a1a;'>"
        "<b>Ecuacion del Cambio Evolutivo:</b><br><br>"
        "<center><b>Dz = Cov(w, z) / w_mean</b></center><br>"
        "Si Cov &gt; 0: z alto da mas descendientes, z_mean aumenta<br>"
        "Si Cov &lt; 0: z bajo da mas descendientes, z_mean disminuye<br>"
        "Si Cov = 0: sin cambio direccional<br><br>"
        "Heredabilidad perfecta asumida (h2 = 1)"
        "</div>"
    )

    def get_params():
        ft = fit_dd.value
        if ft == 'linear_positive':
            return {'a': la_sl.value, 'b': lb_sl.value}
        elif ft == 'linear_negative':
            return {'a': la_sl.value, 'b': -abs(lb_sl.value)}
        elif ft == 'quadratic':
            return {'a': 2.0, 'z_opt': qo_sl.value, 'width': qw_sl.value}
        return {}

    def update_plot():
        with output:
            output.clear_output(wait=True)
            fig, axes = plt.subplots(1, 3, figsize=(16, 5))
            sim.plot_current_state(fit_dd.value, get_params(), axes)
            plt.tight_layout(); plt.show()
            sim.w = sim.fitness_function(sim.z, fit_dd.value, get_params())
            st = sim.compute_evolution_equation()
            print("Generacion", sim.generation,
                  "| z_mean =", round(st['z_mean'], 4),
                  "| w_mean =", round(st['w_mean'], 4),
                  "| Cov =", round(st['cov_wz'], 4),
                  "| Dz_pred =", round(st['delta_z_predicted'], 4))
            if st['cov_wz'] > 0.001:
                print("=> rasgo AUMENTARA en la siguiente generacion")
            elif st['cov_wz'] < -0.001:
                print("=> rasgo DISMINUIRA en la siguiente generacion")
            else:
                print("=> sin cambio direccional")

    def on_step(b):
        sim.w = sim.fitness_function(sim.z, fit_dd.value, get_params())
        pred, actual = sim.selection_step()
        update_plot()
        print("Predicho:", round(pred, 4), "  Observado:", round(actual, 4))

    def on_run_n(n):
        for _ in range(n):
            sim.w = sim.fitness_function(sim.z, fit_dd.value, get_params())
            sim.selection_step()
        update_plot()

    def on_fit_change(change):
        ft = change['new']
        lin_box.layout.visibility  = 'visible' if 'linear'    in ft else 'hidden'
        quad_box.layout.visibility = 'visible' if ft == 'quadratic' else 'hidden'
        update_plot()

    fit_dd.observe(on_fit_change, names='value')
    for sl in [la_sl, lb_sl, qo_sl, qw_sl]:
        sl.observe(lambda c: update_plot(), names='value')

    step_btn.on_click(on_step)
    run10_btn.on_click(lambda b: on_run_n(10))
    run50_btn.on_click(lambda b: on_run_n(50))
    reset_btn.on_click(lambda b: (sim.reset_population(), update_plot()))

    lin_box  = widgets.VBox([la_sl, lb_sl])
    quad_box = widgets.VBox([qo_sl, qw_sl])
    quad_box.layout.visibility = 'hidden'

    controls = widgets.VBox([
        info, fit_dd, lin_box, quad_box,
        widgets.HTML("<hr>"),
        widgets.HBox([step_btn, run10_btn]),
        widgets.HBox([run50_btn, reset_btn])
    ])
    update_plot()
    display(widgets.HBox([controls, output]))


# ============================================================
# CONFIRMACION
# ============================================================
display(HTML(
    "<div style='background-color:#c8e6c9;padding:20px;border-radius:10px;"
    "margin:20px 0;color:#1a1a1a;'>"
    "<h2 style='color:#2e7d32;'>Simuladores Cargados Exitosamente</h2>"
    "<p>Ejecuta cada celda de abajo para lanzar el simulador correspondiente.</p>"
    "</div>"
))
print("Sim 1: create_evolutionary_algorithm_interactive()")
print("Sim 2: create_contagion_interactive()")
print("Sim 3: create_covariance_interactive()")
print("Sim 4: create_evolutionary_change_interactive()")


In [ ]:
create_evolutionary_algorithm_interactive()   # Simulador 1: Algoritmo Evolutivo

In [ ]:
create_contagion_interactive()                # Simulador 2: Ecuaciones en Diferencia

In [ ]:
create_covariance_interactive()               # Simulador 3: Covarianza

In [ ]:
create_evolutionary_change_interactive()      # Simulador 4: Ecuacion de Cambio Evolutivo